In [1]:
import sys

import pandas as pd

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts

from dice4el.scenario.scenario_handler import ScenarioHandler
from dice4el.scenario.scenario_model import ScenarioLSTM, train_ScenarioLSTM, validate_ScenarioLSTM

### --- Load Dataset ---

In [2]:
set_seed(seed=42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
df = pd.read_excel(
    "../../../data/bpic17.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "case:LoanGoal": "string",
        "case:ApplicationType": "string",
        "Accepted": "string",
        "Selected": "string",
        "case:RequestedAmount": "float32",
        "FirstWithdrawalAmount": "float32",
        "NumberOfTerms": "float32",
        "MonthlyCost": "float32",
        "CreditScore": "float32",
        "OfferedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,Accepted,CreditScore,FirstWithdrawalAmount,MonthlyCost,NumberOfTerms,OfferedAmount,Selected,case:ApplicationType,case:LoanGoal,case:RequestedAmount,concept:name,lifecycle:transition,org:resource,time_delta
0,Application_1000086665,2016-08-03 15:57:21.673,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Create Application,complete,User_1,0.000000e+00
1,Application_1000086665,2016-08-03 15:57:21.734,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Submitted,complete,User_1,6.100000e-02
2,Application_1000086665,2016-08-03 15:58:28.299,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Concept,complete,User_1,6.656500e+01
3,Application_1000086665,2016-08-05 13:57:07.419,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Accepted,complete,User_5,1.655191e+05
4,Application_1000086665,2016-08-05 13:59:57.320,True,0.0,5000.0,241.279999,22.0,5000.0,False,New credit,"Other, see explanation",5000.0,O_Create Offer,complete,User_5,1.699010e+02
5,Application_1000086665,2016-08-05 13:59:58.162,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Created,complete,User_5,8.420000e-01
6,Application_1000086665,2016-08-05 14:01:23.264,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Sent (mail and online),complete,User_5,8.510200e+01
7,Application_1000086665,2016-08-05 14:01:23.288,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Complete,complete,User_5,2.400000e-02
8,Application_1000086665,2016-09-05 06:00:36.710,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Cancelled,complete,User_1,2.649554e+06
9,Application_1000086665,2016-09-05 06:00:36.829,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Cancelled,complete,User_1,1.190000e-01


In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['Accepted', 'CreditScore', 'FirstWithdrawalAmount', 'MonthlyCost', 'NumberOfTerms', 'OfferedAmount', 'Selected', 'case:ApplicationType', 'case:LoanGoal', 'case:RequestedAmount', 'concept:name', 'lifecycle:transition', 'org:resource', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
case:LoanGoal                  categorical    case     yes    ['Boat', 'Business goal', 'Car', ...]    N/A        data_derived        
case:ApplicationType           categorical    case     yes    ['Limit raise', 'New credit']            N/A        data_derived        
Accepted         

In [7]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

### --- Scenario Model ---

In [8]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [9]:
scenario_df = scenario_handler.generate_scenario_df(
    df=df,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
    n_scenarios_per_length=3
)

In [10]:
scenario_df.head()

,case:concept:name,time_index,fake,Accepted,CreditScore,FirstWithdrawalAmount,MonthlyCost,NumberOfTerms,OfferedAmount,Selected,case:ApplicationType,case:LoanGoal,case:RequestedAmount,concept:name,lifecycle:transition,org:resource,time_delta
0,f_55597,0,True,True,725.527123,3107.702496,503.613352,119.935582,46524.146810,True,New credit,Caravan / Camper,29615.634012,A_Create Application,complete,User_96,655.213548
1,f_55597,1,True,True,702.011904,24841.497648,554.406999,77.852179,10758.940774,False,New credit,Caravan / Camper,29615.634012,O_Sent (online only),complete,User_63,0.280830
2,f_55597,2,True,False,747.105234,18786.481363,388.947313,123.953329,35257.673280,True,New credit,Caravan / Camper,29615.634012,O_Create Offer,complete,User_20,5467.537991
3,f_55597,3,True,False,558.592039,27170.046191,561.767739,105.720650,46408.384950,False,New credit,Caravan / Camper,29615.634012,O_Sent (online only),complete,User_77,0.054430
4,f_55597,4,True,False,534.944644,2424.642582,620.686842,120.696394,37565.833014,True,New credit,Caravan / Camper,29615.634012,A_Concept,complete,User_70,0.163860


In [11]:
case_ids = scenario_df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = scenario_df[scenario_df["case:concept:name"].isin(train_cases)].copy()
val_df   = scenario_df[scenario_df["case:concept:name"].isin(val_cases)].copy()

In [12]:
train_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=train_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [13]:
val_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=val_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [15]:
criterion = torch.nn.BCEWithLogitsLoss()

In [16]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic17-scenario_model_output.txt")

Epoch 020/100 | Train Loss: 0.0000 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.0000 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.0000 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.0000 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.0000 | LR: 1.00e-06
Time taken for scenario model (training): 12621.771442 seconds
Time taken for scenario model (validation): 8.640183 seconds
Val loss: {'loss': 1.0263624832162163e-12, 'accuracy': 1.0, 'f1_macro': 1.0, 'f1_weighted': 1.0}


In [17]:
embedding_metadata = scenario_handler.get_scenario_embedding_metadata()

scenario_model = ScenarioLSTM(
    categorical_info=embedding_metadata["categorical_info"],
    n_continuous=embedding_metadata["n_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ScenarioLSTM(
    model=scenario_model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

scenario_model.save()

In [18]:
scenario_model = ScenarioLSTM.load()

In [19]:
val_loss = validate_ScenarioLSTM(
    model=scenario_model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [20]:
sys.stdout = original_stdout
log_file.close()